# Lab: Build a Simple BPE Tokenizer from Scratch

## Overview
Byte Pair Encoding (BPE) is a tokenization algorithm that starts with small units and repeatedly combines frequently occurring pairs.

In this lab, we will build a simplified BPE tokenizer using Python.

**Simplifications:**

This is not a production tokenizer.

We will:

- use functions instead of a class;
- use UTF-8 bytes as the initial tokens;
- use Counter to count frequent pairs;
- use a small training dataset;
- ignore performance;
- ignore special tokens;
- ignore caching;
- ignore GPT-2 vocabulary files;
- focus on understanding the algorithm.

## Part 1: Text to UTF-8 Bytes

In [1]:
text = "Building a simplified BPE tokenizer."

# Python can convert a string into UTF-8 bytes:
byte_array = bytearray(text, "utf-8")

print(byte_array)   # b indicates that we are looking into bytes.

bytearray(b'Building a simplified BPE tokenizer.')


In [2]:
# We can convert the byte values into integers
ids = list(byte_array)
print(text)
print(ids)    # Each number represents one byte.

Building a simplified BPE tokenizer.
[66, 117, 105, 108, 100, 105, 110, 103, 32, 97, 32, 115, 105, 109, 112, 108, 105, 102, 105, 101, 100, 32, 66, 80, 69, 32, 116, 111, 107, 101, 110, 105, 122, 101, 114, 46]


### Why start with bytes?
A byte can have a value from 0 to 255.

Therefore, our initial vocabulary can contain 256 possible byte tokens.

Later, BPE will create additional tokens.

## Task 1: Write text_to_bytes()

Create a function called text_to_bytes().

The function should:

1. Receive a Python string.
2. Convert it to UTF-8 bytes.
3. Return those bytes as a list of integers.

In [3]:
def text_to_bytes(text):
  # convert text string to UTF-8 bytes (0 to 255)
  byte_array = bytearray(text, "utf-8")

  # convert the byte values into list of integers and return the list
  return list(byte_array)


In [4]:
text = "the cat has a hat!"
print(text_to_bytes(text))    # initial tokens (bytes 0 to 255)

[116, 104, 101, 32, 99, 97, 116, 32, 104, 97, 115, 32, 97, 32, 104, 97, 116, 33]


## Checkpoint 1

1. What does UTF-8 do?

- UTF-8 ia a way of encoding text so that computers can store and understand characters correctly. UTF-8 bytes converts the text into actual numbers from 0 to 255.

2. What is the token ID for the character t in our example?

- 116

3. Why are the initial IDs between 0 and 255?

- Because bytes can only have value from 0 to 255.

4. What does text_to_bytes(text) return?
- The function will take an argument of strings and returns the list of numbers that were converted into byte values.

## Part 2: Finding Neighboring Pairs

In [5]:
tokens = [1, 2, 3, 2, 3]

pairs = list(zip(tokens, tokens[1:]))
print(pairs)

[(1, 2), (2, 3), (3, 2), (2, 3)]


### Counting Pairs

In [6]:
from collections import Counter

pair_counts = Counter(pairs)

print(pair_counts)

Counter({(2, 3): 2, (1, 2): 1, (3, 2): 1})


In [7]:
most_common = pair_counts.most_common(1)[0][0]

print(most_common)

(2, 3)


## Task 2: Write get_frequent_pair()

In [8]:
def get_frequent_pair(tokens):
  token_pairs = list(zip(tokens, tokens[1:]))
  pair_counts = Counter(token_pairs)

  if not pair_counts:
      return None

# unpacking the tuple
  most_common, count = pair_counts.most_common(1)[0]

  return None if count < 2 else most_common

In [9]:
tokens = [1, 2, 3, 2, 3, 2, 3]

pair = get_frequent_pair(tokens)

print("Most frequent pair:", pair)

Most frequent pair: (2, 3)


In [10]:
# test edge case
print(get_frequent_pair([0]))

None


### Checkpoint 2

Explain the following line:

`pairs = list(zip(tokens, tokens[1:]))`
The line above returns pairs of the tokens starting from index 1 until the end of the list.

Then explain why we use:

`Counter(pairs)`
The line above counts how frequently each pair occurs.

## Part 3: Replacing a Pair

In [11]:
def replace_pair(tokens, pair, new_id):
    result = []
    i = 0

    while i < len(tokens):

        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            result.append(new_id)
            i += 2

        else:
            result.append(tokens[i])
            i += 1

    return result

In [12]:
tokens = [1, 2, 3, 2, 3, 4]

new_tokens = replace_pair(tokens, (2, 3), 256)

print("Before:", tokens)
print("After :", new_tokens)

Before: [1, 2, 3, 2, 3, 4]
After : [1, 256, 256, 4]


### Checkpoint 3

Why do we use `i += 2` when a pair is found?


- We use `i += 2` because a matching pair contains 2 tokens, and both have already been replaced by the argument value of new_id. Moving forward by 2 skips over the entire pair.

What would happen if we used `i += 1`
instead?

- If we used `i += 1`, the loop would move only past the first token, so the second token could participate in another check, which can cause overlapping or incorrect replacements.

## Part 4: Training BPE

## Task 4: Write `train_bpe()`

In [20]:
def train_bpe(tokens, num_merges):
  # create an empty merges dictionary
  merges = {}

  # Make a copy of token so we do not modify the caller's list.
  tokens = tokens.copy()

  # New token IDs start after the 256 possible byte values.
  for new_id in range(256, 256 + num_merges):

    # Find the most frequent neighboring pair.
    pair = get_frequent_pair(tokens)

    # Stop if no pair can be formed
    if pair is None:
      break

    # Replace the pair with the new token ID
    tokens = replace_pair(tokens, pair, new_id)

    # Remember the merge rule
    merges[pair] = new_id

  return merges

**Calling `train_bpe()`**

In [21]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

print("Training tokens:", training_tokens)

Training tokens: [116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]


**Train the tokenizer**

In [22]:
merges = train_bpe(training_tokens, num_merges=10)

print("Learned merges:")

for pair, new_id in merges.items():
  print(pair, "->", new_id)

Learned merges:
(116, 104) -> 256
(256, 101) -> 257
(257, 32) -> 258
(97, 116) -> 259


### Making Training Observable

In [23]:
def train_bpe_observable(tokens, num_merges):

  merges = {}
  tokens = tokens.copy()

  for new_id in range(256, 256 + num_merges):
    pair = get_frequent_pair(tokens)

    if pair is None:
      break

    tokens = replace_pair(tokens, pair, new_id)

    merges[pair] = new_id

    print("Merge:", pair, "→", new_id)
    print("Tokens:", tokens)
    print()

  return merges

In [24]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

merges = train_bpe_observable(training_tokens, num_merges=10)


Merge: (116, 104) → 256
Tokens: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116]

Merge: (256, 101) → 257
Tokens: [257, 32, 99, 97, 116, 32, 105, 110, 32, 257, 32, 104, 97, 116]

Merge: (257, 32) → 258
Tokens: [258, 99, 97, 116, 32, 105, 110, 32, 258, 104, 97, 116]

Merge: (97, 116) → 259
Tokens: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]



### Checkpoint 4

1. Explain the difference between `get_frequent_pair(tokens)` and `replace_pair(tokens, pair, new_id)`.

`get_frequent_pair(tokens)` function finds the most frequently occurring neighboring token pair in the token sequence. `replace_pair(tokens, pair, new_id)` takes a specific pair and replaces every occurrence of that pair with the given new token ID.


2. Explain what information is stored in `merges`.
The `merges` dictionary stores the BPE merge rules learned during training. It records each merged token pair and the new token ID assigned to that pair.

# Part 5: Building a Vocabulary

In [27]:
vocab = {i: bytes([i]) for i in range(256)}

print(vocab)

print(vocab[116])
print(vocab[104])
print(vocab[101])

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

## Task 5: Modify Training to Build the Vocabulary

In [29]:
def train_bpe(tokens, num_merges):

  merges = {}

  # Create the initial vocabulary.
  # IDs 0-255 represent individual byte values.
  vocab = {i: bytes([i]) for i in range(256)}

  tokens = tokens.copy()

  # Create new token IDs starting from 256.
  for new_id in range(256, 256 + num_merges):
    pair = get_frequent_pair(tokens)

    if pair is None:
      break

    # Merge the pair
    tokens = replace_pair(tokens, pair, new_id)

    # Remember the merge rule
    merges[pair] = new_id

    # Build the byte representation of the new token
    vocab[new_id] = (vocab[pair[0]] + vocab[pair[1]])

  return merges, vocab

## Calling the New `train_bpe()`

In [32]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

merges, vocab = train_bpe(training_tokens, num_merges=10)

In [46]:
print("Number of merges:", len(merges))
print("Vocabulary size:", len(vocab))

for token_id in sorted(vocab):
  if token_id >= 256:
    print(token_id, "->", vocab[token_id])

Number of merges: 4
Vocabulary size: 260
{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W'

### Checkpoint 5
1. Why does the vocabulary contain 0–255 from the beginning?
The vocabulary contains IDs 0–255 because a byte can have 256 possible values, from 0 to 255.

2. Why do new tokens start at 256?
New tokens start at ID 256 because IDs 0–255 are already used for the 256 possible byte values. The first newly created BPE token therefore gets ID 256.

3. What does an entry such as `vocab[256]` represent?
`vocab[256]` represents the token associated with token ID 256. It is a dictionary lookup that tells us what byte sequence or token was created by that BPE merge.



# Part 6: Encoding Text

## Task 6: Write `encode()`

In [36]:
def encode(text, merges):
  tokens = text_to_bytes(text)

  for pair, new_id in merges.items():
    tokens = replace_pair(tokens, pair, new_id)

  return tokens

**Calling `encode()`**

In [40]:
text = "the cat in the hat"

ids = encode(text, merges)

print("Text:", text)
print("Token IDs:", ids)

print()
print({"the": 258}, {" c": 99}, {"at": 259}, {" ": 32}, {"i": 105}, {"n": 110}, {" ": 32}, {"the": 258}, {" h": 104}, {"at": 259})

Text: the cat in the hat
Token IDs: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]

{'the': 258} {' c': 99} {'at': 259} {' ': 32} {'i': 105} {'n': 110} {' ': 32} {'the': 258} {' h': 104} {'at': 259}


# Part 7: Decoding

## Task 7: Write `decode()`

In [42]:
def decode(token_ids, vocab):
  # Start with an empty byte string
  result = b""

  # Look up every token
  for token_id in token_ids:
    result += vocab[token_id]

  # Convert the final bytes back into text
  return result.decode("utf-8")

**Calling `decode()`**

In [43]:
ids = encode("the cat in the hat", merges)

decoded_text = decode(ids, vocab)

print("Encoded:", ids)
print("Decoded:", decoded_text)

Encoded: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]
Decoded: the cat in the hat


In [45]:
# If nothing is printed, the assertion passed.
assert decoded_text == "the cat in the hat"

### Checkpoint 7

1. Explain why decoding does not need to manually perform:

```text
257
 ↓
256 + 101
 ↓
116 + 104 + 101
```
in our implementation.

- Decoding does not need to manually perform the previous merge steps because the vocabulary already stores the complete byte sequence represented by each token ID.

2. What does the vocabulary allow us to do instead?
- The vocabulary allows us to look up each token ID, retrieve its corresponding byte sequence, concatenate the byte sequences, and then decode the resulting bytes back into text.

# Part 8: Seeing the Complete Training Process

In [47]:
def train_bpe_observable(tokens, num_merges):
    merges = {}

    vocab = {i: bytes([i])for i in range(256)}

    tokens = tokens.copy()

    for new_id in range(256, 256 + num_merges):

        pair = get_frequent_pair(tokens)

        if pair is None:
            break

        tokens = replace_pair(tokens, pair, new_id)

        merges[pair] = new_id

        vocab[new_id] = (vocab[pair[0]] + vocab[pair[1]])

        print(f"Merge: {pair} → {new_id}")

        print(f"Token represents: {vocab[new_id]!r}")

        print(f"Current tokens: {tokens}")

        print()

    return merges, vocab

## Calling Observable Training

In [49]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

merges, vocab = train_bpe_observable(training_tokens,num_merges=10)

Merge: (116, 104) → 256
Token represents: b'th'
Current tokens: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116]

Merge: (256, 101) → 257
Token represents: b'the'
Current tokens: [257, 32, 99, 97, 116, 32, 105, 110, 32, 257, 32, 104, 97, 116]

Merge: (257, 32) → 258
Token represents: b'the '
Current tokens: [258, 99, 97, 116, 32, 105, 110, 32, 258, 104, 97, 116]

Merge: (97, 116) → 259
Token represents: b'at'
Current tokens: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]



# Part 9: Complete BPE Pipeline

The complete process is:

                 TRAINING
                    │
                    ↓
             Training text
                    │
                    ↓
              UTF-8 bytes
                    │
                    ↓
          Find frequent pairs
                    │
                    ↓
              Merge pairs
                    │
                    ↓
          Learn merge rules
                    │
                    ↓
              Build vocab
                    │
                    │
                    ▼
                  ENCODE
                    │
                    ↓
              New text
                    │
                    ↓
              UTF-8 bytes
                    │
                    ↓
           Apply learned merges
                    │
                    ↓
                Token IDs
                    │
                    ▼
                  DECODE
                    │
                    ↓
                 Bytes
                    │
                    ↓
                  Text

## Task 9: Run the Complete Pipeline

In [50]:
# Provide text
text = "the cat in the hat"

In [64]:
# Convert text to bytes 0-255
tokens = text_to_bytes(text)

print("Initial tokens:", tokens)

Initial tokens: [116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]


In [52]:
# Train the tokenizer
merges, vocab = train_bpe(tokens, num_merges=10)

In [54]:
# Encode token IDs
ids = encode(text, merges)

print("Token IDs:", ids)

Token IDs: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]


In [55]:
# Decode token IDs
decoded = decode(ids, vocab)

print("Decoded text:", decoded)

Decoded text: the cat in the hat


## Final Test

In [57]:
print("Original :", text)
print("Token IDs:", ids)
print("Decoded  :", decoded)

assert decoded == text #Empty output means success

Original : the cat in the hat
Token IDs: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]
Decoded  : the cat in the hat


# Part 10: Experiment with the Tokenizer

### Experiment 1: Change the number of merges

In [63]:
merges, vocab = train_bpe(tokens, num_merges=20)
ids = encode(text, merges)

print("Token IDs:", ids)

Token IDs: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]


**Observation**

More merges can change tokenization, but only when the learned merge rules apply to the text being encoded.

### Experiment 2: Change the training text

In [71]:
training_text = "hello hello hello"
training_tokens = text_to_bytes(training_text)

merges, vocab = train_bpe(training_tokens, num_merges=10)

for pair, new_id in merges.items():
    print(pair, "→", new_id, "=", vocab[new_id])

print()
print(tokens)

(104, 101) → 256 = b'he'
(256, 108) → 257 = b'hel'
(257, 108) → 258 = b'hell'
(258, 111) → 259 = b'hello'
(259, 32) → 260 = b'hello '

[104, 101, 108, 108, 111, 32, 104, 101, 108, 108, 111, 32, 104, 101, 108, 108, 111]


### Experiment 3: Encode new text

In [73]:
new_text = "the hat"

ids = encode(new_text, merges)

print(ids)

[116, 256, 32, 104, 97, 116]


In [74]:
decoded = decode(ids, vocab)

print(decoded)

the hat


# Part 11: Understanding the Three Main Functions

At this point, the most important functions are:

## `text_to_bytes()`

Converts text into initial byte tokens.

```text
"the"
 ↓
[116, 104, 101]
```

---

## `train_bpe()`

Learns merge rules from training data.

```text
[116, 104, 101, ...]
        ↓
find frequent pair
        ↓
create new token
        ↓
record merge
        ↓
repeat
```

---

## `encode()`

Uses the learned rules on text.

```text
"the cat"
   ↓
byte IDs
   ↓
learned merges
   ↓
token IDs
```

---

## `decode()`

Converts token IDs back into text.

```text
token IDs
   ↓
vocabulary lookup
   ↓
bytes
   ↓
UTF-8
   ↓
text
```

---

# Part 12: Complete Script

In [75]:
from collections import Counter


# --------------------------------------------------
# 1. Convert text to UTF-8 byte IDs
# --------------------------------------------------

def text_to_bytes(text):
    """
    Convert a Python string into a list of UTF-8 byte IDs.

    Each byte has a value from 0 to 255.
    """
    byte_array = bytearray(text, "utf-8")

    return list(byte_array)


# --------------------------------------------------
# 2. Find the most frequent neighboring pair
# --------------------------------------------------

def get_frequent_pair(tokens):
    """
    Find and return the most frequent neighboring pair.

    Example:

        [1, 2, 3, 2, 3]

    produces:

        (1, 2)
        (2, 3)
        (3, 2)
        (2, 3)

    The most frequent pair is:

        (2, 3)
    """

    if len(tokens) < 2:
        return None

    pairs = list(zip(tokens, tokens[1:]))

    pair_counts = Counter(pairs)

    return pair_counts.most_common(1)[0][0]


# --------------------------------------------------
# 3. Replace a pair with a new token ID
# --------------------------------------------------

def replace_pair(tokens, pair, new_id):
    """
    Replace every occurrence of 'pair' with 'new_id'.

    Example:

        tokens = [1, 2, 3, 2, 3, 4]
        pair = (2, 3)
        new_id = 256

    result:

        [1, 256, 256, 4]
    """

    result = []

    i = 0

    while i < len(tokens):

        if (i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair):
            result.append(new_id)

            # We consumed two tokens.
            i += 2

        else:
            result.append(tokens[i])

            # We consumed one token.
            i += 1

    return result


# --------------------------------------------------
# 4. Train the BPE tokenizer
# --------------------------------------------------

def train_bpe(tokens, num_merges):
    """
    Learn BPE merge rules from training tokens.

    Returns:

        merges
        vocab
    """

    merges = {}

    # Initial vocabulary contains all 256 byte values.
    vocab = {
        i: bytes([i])
        for i in range(256)
    }

    # Work on a copy of the training tokens.
    tokens = tokens.copy()

    # New token IDs start at 256.
    for new_id in range(256, 256 + num_merges):

        # Find the most frequent pair.
        pair = get_frequent_pair(tokens)

        # Stop if there are no pairs.
        if pair is None:
            break

        # Replace the pair with the new token ID.
        tokens = replace_pair(tokens, pair, new_id)

        # Remember the merge rule.
        merges[pair] = new_id

        # Build the byte representation of the new token.
        vocab[new_id] = (vocab[pair[0]] + vocab[pair[1]])

    return merges, vocab


# --------------------------------------------------
# 5. Encode text
# --------------------------------------------------

def encode(text, merges):
    """
    Convert text into token IDs using learned merges.
    """

    # Start with byte IDs.
    tokens = text_to_bytes(text)

    # Apply learned merges in order.
    for pair, new_id in merges.items():

        tokens = replace_pair(tokens, pair, new_id)

    return tokens


# --------------------------------------------------
# 6. Decode token IDs
# --------------------------------------------------

def decode(token_ids, vocab):
    """
    Convert token IDs back into a Python string.
    """

    result = b""

    for token_id in token_ids:

        result += vocab[token_id]

    return result.decode("utf-8")


# ==================================================
# EXECUTE THE COMPLETE PIPELINE
# ==================================================

# Training data
training_text = "the cat in the hat"

# Step 1: Convert training text to byte tokens
training_tokens = text_to_bytes(training_text)

print("Training tokens:")
print(training_tokens)
print()


# Step 2: Train BPE
merges, vocab = train_bpe(
    training_tokens,
    num_merges=10
)

print("Learned merges:")
for pair, new_id in merges.items():
    print(pair, "→", new_id)

print()


# Step 3: Encode text
ids = encode(training_text, merges)

print("Encoded IDs:")
print(ids)
print()


# Step 4: Decode text
decoded = decode(ids, vocab)

print("Decoded text:")
print(decoded)
print()


# Step 5: Verify correctness
assert decoded == training_text

print("Encoding and decoding successful!")

Training tokens:
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]

Learned merges:
(116, 104) → 256
(256, 101) → 257
(257, 32) → 258
(97, 116) → 259
(258, 99) → 260
(260, 259) → 261
(261, 32) → 262
(262, 105) → 263
(263, 110) → 264
(264, 32) → 265

Encoded IDs:
[265, 258, 104, 259]

Decoded text:
the cat in the hat

Encoding and decoding successful!


# Part 13: Compare with `tiktoken`

In [76]:
!pip install tiktoken --q

In [77]:
import tiktoken

In [78]:
tokenizer = tiktoken.get_encoding("gpt2")

In [80]:
text = "GPT2 is providing encoding"

ids = tokenizer.encode(text)

print(ids)

[38, 11571, 17, 318, 4955, 21004]


In [82]:
decoded = tokenizer.decode(ids)

print(decoded)

GPT2 is providing encoding


# Final Questions
Answer these questions after completing the lab.

**Question 1**

Why do we start with 256 possible byte token IDs?
- Because a byte can represent 256 different values from 0 to 255. these 256 byte values form the initial vocabulary of the byte-level BPE tokenizer.

**Question 2**

Why does the first newly created token have ID 256?

- As the IDs 0 to 255 are already reserved for the original byte values, the next available ID is 256.

**Question 3**

What does the following merge mean?

(116, 104) → 256

- The merge above means that the byte pair (116, 104) ("th") appeared frequently during training, so BPE created a new token with the ID 256 to represent that pair.

**Question 4**

What is the difference between BPE training and encoding?

- BPE training learns the most frequent byte pairs and creates new tokens by building merge rules and a vocabulary. Encoding does not learn anything. It simply applies the previously learned merge rules to convert raw text into token IDs.

**Question 5**

Why do we need to store the learned merges?

- We store the learned merges so the tokenizer can apply the same merge rules when encoding new text. without the stored merges, the tokenizer would have to retrain every time it processes text.

**Question 6**

Why do we need a vocabulary?

- We need a vocabulary because it maps each token ID to the byte sequence it represents. This allows the tokenizer to convert token IDs back into bytes and decode them into the original text.

**Question 7**

What does decode() do?

- `decode()` converts token IDs back into their corresponding byte sequences using the vocabulary, concatenates those bytes, and then decodes them into readable text.

**Question 8**

What happens when we increase num_merges?

- Increasing `num_merges` allows the tokenizer to learn more merge rules and create more tokens. This can reduce the number of tokens needed to represent text. However, if no additional useful merges are found or the extra merges do not apply to the encoded text, the output may remain unchanged.

**Question 9**

Why can the tokenizer trained on:

"the cat in the hat"
also encode:

"the hat"
?

- The tokenizer can encode "the hat" because the necessary merge rules and vocabulary were already learned during training. Encoding simply reuses those learned rules to tokenize new text that containes familiar patterns.

**Question 10**

What are some differences between this educational tokenizer and a production tokenizer such as `tiktoken`?

- This educational tokenizer is a simplified implementation designed to demonstrate how BPE works. It trains on a very small dataset, performs a limited number of merges, and uses basic Python data structures. Production tokenizers such as `tiktoken` are highly optimized, trained on massive corpora, use sophisticated merge rules and vocabularies, support many languages and Unicode characters, are much faster, and are designed to work efficiently with large language models such as GPT.